# Section 4 · Notebook 2 — Agent as Tool: Tutoring Assistant

A teacher agent receives a student's question and delegates to the right specialist tutor (math, writing, science). The teacher always stays in control.

In [1]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from strands.models.bedrock import BedrockModel

MODEL_ID = os.getenv("BEDROCK_MODEL_ID", "global.anthropic.claude-sonnet-4-6")
REGION = os.getenv("AWS_REGION", os.getenv("AWS_DEFAULT_REGION", "us-east-1"))
os.environ.setdefault("AWS_REGION", REGION)

model = BedrockModel(model_id=MODEL_ID, region_name=REGION)
print(f"Using {MODEL_ID} in {REGION}")

Using global.anthropic.claude-sonnet-4-6 in us-east-1


In [3]:
from strands import Agent

## Specialist agents

Each specialist gets a `name` and a `description`. When an agent is used as a tool, the name becomes the tool name and the description tells the teacher when to call it.

In [4]:
math_tutor = Agent(
    name="math_tutor",
    description="Solves mathematical equations and explains math concepts step by step",
    model=model,
    system_prompt="You are a math tutor. Solve math problems and explain your solutions clearly, step by step.",
    callback_handler=None,
)

writing_coach = Agent(
    name="writing_coach",
    description="Helps with writing, grammar, and essay structure",
    model=model,
    system_prompt="You are a writing coach. Help with writing, grammar, and essay structure. Provide constructive feedback.",
    callback_handler=None,
)

science_helper = Agent(
    name="science_helper",
    description="Explains scientific concepts and principles in simple terms",
    model=model,
    system_prompt="You are a science tutor. Explain scientific concepts clearly using simple language and real-world examples.",
    callback_handler=None,
)

## The teacher (orchestrator)

Pass the specialist agents straight into `tools=[...]` — Strands wraps each one as a callable tool.

In [5]:
teacher = Agent(
    model=model,
    system_prompt="""You are a teacher who helps students by delegating to specialist tutors.
Never solve problems yourself. Always delegate to the appropriate specialist:
- math_tutor: for math problems and equations
- writing_coach: for writing, grammar, and essays
- science_helper: for science questions

After receiving the specialist's response, present it to the student in a friendly way.""",
    tools=[math_tutor, writing_coach, science_helper],
)

## Test with different questions

In [6]:
# Math → math_tutor
response = teacher("Can you help me solve this quadratic equation: x^2 + 5x + 6 = 0?")

Sure! Let me pass your equation to our math specialist right away! 🎓
Tool #1: math_tutor
Great news — our math tutor has solved it for you! Here's a breakdown of the solution:

### 🧮 Solving x² + 5x + 6 = 0

**Step 1: Factor the equation**
The key is finding two numbers that **multiply to 6** and **add to 5**. Those numbers are **2 and 3**!
So the equation factors into: **(x + 2)(x + 3) = 0**

**Step 2: Apply the Zero Product Property**
If two things multiply to zero, at least one of them must be zero:
- x + 2 = 0 → **x = -2**
- x + 3 = 0 → **x = -3**

**✅ The two solutions are x = -2 and x = -3**

Both answers were also verified by plugging them back into the original equation — and they check out perfectly! 🎉

Let me know if you'd like a deeper explanation of any step, or if you have more problems to solve! 😊

In [7]:
# Writing → writing_coach
response = teacher("How do I write a strong thesis statement for my essay about climate change?")

Great question! Let me connect you with our writing specialist right away! ✍️
Tool #2: writing_coach
Awesome — our writing coach has some fantastic guidance for you! Here's what you need to know:

### ✍️ Writing a Strong Thesis Statement for Climate Change

**A great thesis must:**
- Make a **specific, arguable claim** (not just a fact)
- **Signal your essay's direction** clearly
- Take a **clear position** that someone could reasonably debate
- Be **specific yet developed** enough to explore

---

### ❌ Common Mistakes to Avoid
| Weak Example | Why It Fails |
|---|---|
| "Climate change is real." | Too obvious — no argument there! |
| "Climate change affects many things." | Way too vague |
| "This essay will discuss climate change." | That's an announcement, not a thesis! |

---

### ✅ The Strong Thesis Formula
**[Specific topic] + [Your clear claim] + [Primary reasoning]**

Here are two solid examples:
- **Argumentative:** *"Governments must prioritize carbon pricing over individual 

In [8]:
# Science → science_helper
response = teacher("Why does ice float on water?")

Great question! Let me check with our science specialist on that one! 🔬
Tool #3: science_helper
Our science expert has a wonderful explanation for you! 🧊

### Why Does Ice Float on Water?

**It all comes down to density!**
An object floats when it's **less dense** than the liquid it's in — and surprisingly, ice is actually **less dense than liquid water!** This is quite rare in nature!

---

### 🔬 The Molecular Reason
The secret lies in how water molecules behave:

- **In liquid water:** Molecules move around freely and pack together closely.
- **In ice:** Molecules form a **rigid hexagonal crystal structure**, where hydrogen bonds lock them into a more **spread-out arrangement** — taking up *more space* than in liquid form!

> 🧊 Ice is about **9% less dense** than liquid water — which is why only about 1/10 of an iceberg is visible above the surface!

---

### 🌍 Why This Matters for Life on Earth
This isn't just a cool science fact — it's **vital for life!**
- Lakes freeze **from the 